# Credit Card Fraud Detection

## Data dictionary
1.transaction_id :รหัสประจำธุรกรรม ใช้ระบุธุรกรรมแต่ละรายการไม่ให้ซ้ำกัน

2.amount_usd :จำนวนเงินที่ใช้จ่ายในธุรกรรม หน่วยเป็นดอลลาร์สหรัฐ

3.merchant_category :ประเภทของร้านค้า เช่น ร้านอาหาร โรงแรม ช้อปปิ้ง หรือปั๊มน้ำมัน

4.card_type :ประเภทของบัตร เช่น Credit Card, Debit Card หรือ Platinum

5.auth_method :วิธีการยืนยันตัวตน เช่น PIN, OTP, Contactless หรือ 3D Secure

6.channel :ช่องทางที่ใช้ทำธุรกรรม เช่น Online, POS หรือ Mobile Banking

7.device_type :อุปกรณ์ที่ใช้ทำธุรกรรม เช่น Mobile, Desktop หรือ Tablet

8.is_foreign_transaction 	:ระบุว่าธุรกรรมเกิดขึ้นในต่างประเทศหรือไม่ (1 = ใช่, 0 = ไม่ใช่)

9.hours_since_last_txn :จำนวนชั่วโมงที่ผ่านไปตั้งแต่ธุรกรรมครั้งก่อน

10.txn_count_last_24h :จำนวนธุรกรรมที่เกิดขึ้นภายใน 24 ชั่วโมงล่าสุด

11.distance_from_home_km :ระยะทางจากบ้านของลูกค้าถึงสถานที่ทำธุรกรรม หน่วยเป็นกิโลเมตร

12.card_age_months :ระยะเวลาที่ลูกค้าใช้บัตรใบนี้มาแล้ว หน่วยเป็นเดือน

13.customer_age :อายุของลูกค้า

14.account_balance_usd :ยอดเงินคงเหลือในบัญชี หน่วยเป็นดอลลาร์

15.is_new_merchant :ระบุว่าลูกค้าเคยใช้บริการร้านค้านี้มาก่อนหรือไม่ (1 = ร้านค้าใหม่, 0 = ร้านค้าเดิม)

16.used_vpn :ระบุว่ามีการใช้ VPN ในการทำธุรกรรมหรือไม่

17.ip_country_mismatch :ประเทศของ IP Address ไม่ตรงกับประเทศที่คาดหวังหรือข้อมูลตำแหน่งของลูกค้า

18.billing_shipping_mismatch :ที่อยู่สำหรับเรียกเก็บเงินไม่ตรงกับที่อยู่จัดส่งสินค้า

19.cvv_retry_count 	:จำนวนครั้งที่กรอกรหัส CVV ผิดหรือต้องลองใหม่

20.velocity_score :คะแนนที่ใช้วัดความถี่หรือความรวดเร็วของธุรกรรมที่ผิดปกติ

21.time_of_day_hour :ชั่วโมงที่เกิดธุรกรรม เช่น 0–23

22.day_of_week :วันที่เกิดธุรกรรมในหนึ่งสัปดาห์ เช่น Monday–Sunday หรือ 0–6

23.is_ai_generated_scam_attempt :ระบุว่ามีความพยายามหลอกลวงที่สร้างหรือใช้ AI เข้ามาเกี่ยวข้องหรือไม่

24.merchant_risk_score :คะแนนความเสี่ยงของร้านค้า ยิ่งคะแนนสูงอาจหมายถึงร้านค้ามีความเสี่ยงมาก

25.prior_disputes :จำนวนครั้งที่ลูกค้าเคยโต้แย้งหรือแจ้งปัญหาเกี่ยวกับธุรกรรมก่อนหน้านี้

26.is_fraud :ตัวแปรเป้าหมาย (Target Variable) ระบุว่าธุรกรรมนั้นเป็นการฉ้อโกงหรือไม่

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report,
    ConfusionMatrixDisplay, RocCurveDisplay, roc_curve, roc_auc_score
)

os.makedirs("figures", exist_ok=True)

## 1. Data Cleaning

In [ ]:
CreditCardFraud = pd.read_csv('data/creditcardfraud2026.csv')

In [ ]:
CreditCardFraud.head()

In [ ]:
CreditCardFraud.shape

### 1.1 ข้อมูลซ้ำ

In [ ]:
CreditCardFraud.duplicated().sum()

In [ ]:
CreditCardFraud = CreditCardFraud.drop(columns=['transaction_id'])

In [ ]:
CreditCardFraud_duplicated = CreditCardFraud.drop_duplicates()
CreditCardFraud_duplicated

### 1.2 Missing values

In [ ]:
CreditCardFraud_duplicated.isnull().sum()

In [ ]:
target_columns = ['amount_usd',
                  'merchant_category',
                  'device_type',
                  'hours_since_last_txn',
                  'distance_from_home_km',
                  'account_balance_usd',
                  'merchant_risk_score'
                  ]
CreditCardFraud_Missing = CreditCardFraud_duplicated.dropna(subset=target_columns).reset_index(drop=True)
CreditCardFraud_Missing.isnull().sum()

In [ ]:
CreditCardFraud_Missing.shape

### 1.3 ค่าที่ผิดรูปแบบใน amount_usd

In [ ]:
for col in CreditCardFraud_Missing.select_dtypes(include=['object']).columns:
  print(f'{col}: {CreditCardFraud_Missing[col].unique()}')

In [ ]:
CreditCardFraud_Missing.loc[
    pd.to_numeric(CreditCardFraud_Missing['amount_usd'], errors='coerce').isna() &
    CreditCardFraud_Missing['amount_usd'].notna(),
    ['amount_usd']
]

In [ ]:
CreditCardFraud_Missing.loc[CreditCardFraud_Missing['amount_usd'] == 'unknown', 'amount_usd'] = None
CreditCardFraud_Missing.dropna(subset=['amount_usd'], inplace=True)
CreditCardFraud_Missing['amount_usd'] = CreditCardFraud_Missing['amount_usd'].astype(float)
CreditCardFraud_Missing.isnull().sum()

### 1.4 จัดรูปแบบข้อความ (ตัวพิมพ์เล็ก/ใหญ่, ช่องว่าง)

In [ ]:
for col in CreditCardFraud_Missing.select_dtypes(include=['object']).columns:
  print(f'{col}: {CreditCardFraud_Missing[col].unique()}')

In [ ]:
CreditCardFraud_Missing['merchant_category'] = CreditCardFraud_Missing['merchant_category'].str.lower()
CreditCardFraud_Missing['card_type'] = CreditCardFraud_Missing['card_type'].str.lower()

In [ ]:
for col in CreditCardFraud_Missing.select_dtypes(include=['object']).columns:
  print(f'{col}: {CreditCardFraud_Missing[col].unique()}')

In [ ]:
CreditCardFraud_Missing['channel'] = CreditCardFraud_Missing['channel'].str.strip()
CreditCardFraud_Missing['channel'] = CreditCardFraud_Missing['channel'].str.title()
Channel_map = {
    'Pos': 'POS',
    'Atm': 'ATM'
    }
CreditCardFraud_Missing['channel'] = CreditCardFraud_Missing['channel'].replace(Channel_map)

In [ ]:
for col in CreditCardFraud_Missing.select_dtypes(include=['object']).columns:
  print(f'{col}: {CreditCardFraud_Missing[col].unique()}')

In [ ]:
CreditCardFraud_Missing['device_type'] = CreditCardFraud_Missing['device_type'].str.title()
device_type_map = {
    'Pos Terminal': 'POS Terminal' ,
    'Atm Machine': 'ATM Machine' ,
    'Windows Pc': 'Windows PC' ,
    'Iphone': 'iPhone'
    }
CreditCardFraud_Missing['device_type'] = CreditCardFraud_Missing['device_type'].replace(device_type_map)

In [ ]:
for col in CreditCardFraud_Missing.select_dtypes(include=['object']).columns:
  print(f'{col}: {CreditCardFraud_Missing[col].unique()}')

### 1.5 แปลงคอลัมน์ TRUE/FALSE เป็น boolean

In [ ]:
is_new_merchant_map = {
    '1': 'TRUE',
    '0': 'FALSE'
    }
CreditCardFraud_Missing['is_new_merchant'] = CreditCardFraud_Missing['is_new_merchant'].replace(is_new_merchant_map)
used_vpn_map = {
    'Yes': 'TRUE',
    'No': 'FALSE'
    }
CreditCardFraud_Missing['used_vpn'] = CreditCardFraud_Missing['used_vpn'].replace(used_vpn_map)
CreditCardFraud_Missing['is_new_merchant'] = CreditCardFraud_Missing['is_new_merchant'] == 'TRUE'
CreditCardFraud_Missing['used_vpn'] = CreditCardFraud_Missing['used_vpn'] == 'TRUE'

In [ ]:
for col in CreditCardFraud_Missing.select_dtypes(include=['object']).columns:
  print(f'{col}: {CreditCardFraud_Missing[col].unique()}')

### 1.6 ค่าผิดปกติ (Outlier) และบันทึกไฟล์ที่ทำความสะอาดแล้ว

In [ ]:
CreditCardFraud_Cleaning = CreditCardFraud_Missing.copy()
CreditCardFraud_Cleaning.info()

In [ ]:
CreditCardFraud_Cleaning.head()

In [ ]:
CreditCardFraud_Cleaning.describe()

In [ ]:
CreditCardFraud_Outlier = CreditCardFraud_Cleaning[
    (CreditCardFraud_Cleaning['customer_age'] >= 0) & (CreditCardFraud_Cleaning['customer_age'] <= 100) &
    (CreditCardFraud_Cleaning['time_of_day_hour'] >= 0) & (CreditCardFraud_Cleaning['time_of_day_hour'] <= 23) &
    (CreditCardFraud_Cleaning['day_of_week'] >= 0) & (CreditCardFraud_Cleaning['day_of_week'] <= 6) &
    (CreditCardFraud_Cleaning['merchant_risk_score'] >= 0) & (CreditCardFraud_Cleaning['merchant_risk_score'] <= 100)
].copy()
CreditCardFraud_Outlier.describe()

In [ ]:
CreditCardFraud_Cleaning = CreditCardFraud_Outlier.copy()

In [ ]:
CreditCardFraud_Cleaning.to_csv('cleaned_credit_card_fraud.csv', index=False)

## 2. EDA

In [ ]:
df = CreditCardFraud_Cleaning.copy()

df.info()

In [ ]:
df.head()

In [ ]:
plt.figure(figsize=(7, 7))
fraud_counts = df['is_fraud'].value_counts()
colors = ['#3498db', '#f39c12']
plt.pie(fraud_counts, labels=['Normal (0)', 'Fraud (1)'], autopct='%1.2f%%', startangle=90,
        colors=colors, explode=(0, 0.1), shadow=True)
plt.title('1. Proportion of Fraud (is_fraud)', fontsize=12, fontweight='bold')
plt.savefig('figures/eda_1_fraud_proportion.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
plt.figure(figsize=(7, 7))
sns.boxplot(y=df['amount_usd'], color='#3498db')
plt.title('2. Distribution of the amount of money (amount_usd)', fontsize=12, fontweight='bold')
plt.ylabel('Amount (USD)', fontweight='bold')
plt.savefig('figures/eda_2_amount_boxplot.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
channel_counts = df['channel'].value_counts()
sns.barplot(x=channel_counts.index, y=channel_counts.values, palette='viridis',
            hue=channel_counts.index, legend=False)
plt.title('3. Number of transactions by channel (Channel)', fontsize=12, fontweight='bold')
plt.xlabel('Channel', fontweight='bold')
plt.ylabel('Count', fontweight='bold')
plt.xticks(rotation=90)
plt.savefig('figures/eda_3_channel.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
sns.countplot(data=df, x='used_vpn', hue='is_fraud', palette=['#3498db', '#e74c3c'])
plt.title('4. The relationship between the use of VPN and Fraud', fontsize=12, fontweight='bold')
plt.xlabel('Used VPN (False/True)', fontweight='bold')
plt.ylabel('Count', fontweight='bold')
plt.legend(title='Is Fraud', labels=['Normal', 'Fraud'])
plt.savefig('figures/eda_4_vpn_vs_fraud.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
sns.kdeplot(data=df, x='merchant_risk_score', hue='is_fraud', common_norm=False,
            palette=['#2ecc71', '#e74c3c'], fill=True, alpha=0.4)
plt.title('5. Risk score (Merchant Risk Score) Categorized by group Fraud', fontsize=12, fontweight='bold')
plt.xlabel('Merchant Risk Score', fontweight='bold')
plt.ylabel('Density', fontweight='bold')
plt.savefig('figures/eda_5_risk_score.png', dpi=200, bbox_inches='tight')
plt.show()

## 3. เตรียมข้อมูลสำหรับโมเดล

In [ ]:
X = df.drop(columns=["is_fraud"])
y = df["is_fraud"]

X = pd.get_dummies(X, drop_first=True)

print("จำนวนตัวแปรหลังแปลง:", X.shape[1])
print("สัดส่วน fraud:", round(y.mean() * 100, 2), "%")

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

## 4. Decision Tree

In [ ]:
dt_model = DecisionTreeClassifier(max_depth=5, random_state=42, class_weight="balanced")

dt_model.fit(X_train, y_train)

In [ ]:
dt_pred = dt_model.predict(X_test)

print(classification_report(y_test, dt_pred, target_names=["Normal (0)", "Fraud (1)"]))

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test, dt_pred, display_labels=["Normal", "Fraud"], cmap="Blues"
)
plt.title("Decision Tree - Confusion Matrix")
plt.savefig("figures/dt_confusion.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
RocCurveDisplay.from_estimator(dt_model, X_test, y_test, name="Decision Tree")
plt.plot([0, 1], [0, 1], "k--", label="Random Chance")
plt.title("Decision Tree - ROC Curve")
plt.legend(loc="lower right")
plt.grid(True, linestyle="--", alpha=0.5)
plt.savefig("figures/dt_roc.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
dt_importance = pd.Series(dt_model.feature_importances_, index=X.columns)

dt_importance.sort_values().tail(10).plot(kind="barh", color="steelblue")
plt.title("Decision Tree - Top 10 Feature Importance")
plt.xlabel("Importance")
plt.savefig("figures/dt_importance.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
plt.figure(figsize=(22, 10))
plot_tree(
    dt_model, feature_names=X.columns, class_names=["Normal", "Fraud"],
    filled=True, rounded=True, max_depth=3, fontsize=9
)
plt.title("Decision Tree (top 3 levels)")
plt.savefig("figures/dt_tree.png", dpi=200, bbox_inches="tight")
plt.show()

## 5. Random Forest

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=100, max_depth=5, random_state=42, class_weight="balanced"
)

rf_model.fit(X_train, y_train)

In [ ]:
rf_pred = rf_model.predict(X_test)

print(classification_report(y_test, rf_pred, target_names=["Normal (0)", "Fraud (1)"]))

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test, rf_pred, display_labels=["Normal", "Fraud"], cmap="Greens"
)
plt.title("Random Forest - Confusion Matrix")
plt.savefig("figures/rf_confusion.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
RocCurveDisplay.from_estimator(rf_model, X_test, y_test, name="Random Forest")
plt.plot([0, 1], [0, 1], "k--", label="Random Chance")
plt.title("Random Forest - ROC Curve")
plt.legend(loc="lower right")
plt.grid(True, linestyle="--", alpha=0.5)
plt.savefig("figures/rf_roc.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
rf_importance = pd.Series(rf_model.feature_importances_, index=X.columns)

rf_importance.sort_values().tail(10).plot(kind="barh", color="seagreen")
plt.title("Random Forest - Top 10 Feature Importance")
plt.xlabel("Importance")
plt.savefig("figures/rf_importance.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
plt.figure(figsize=(22, 10))
plot_tree(
    rf_model.estimators_[0], feature_names=X.columns, class_names=["Normal", "Fraud"],
    filled=True, rounded=True, max_depth=3, fontsize=9
)
plt.title("Random Forest - Tree 1 (top 3 levels)")
plt.savefig("figures/rf_tree.png", dpi=200, bbox_inches="tight")
plt.show()

## 6. XGBoost

In [ ]:
ratio = (y_train == 0).sum() / (y_train == 1).sum()

xgb_model = XGBClassifier(
    n_estimators=100, max_depth=5, random_state=42, scale_pos_weight=ratio
)

xgb_model.fit(X_train, y_train)

In [ ]:
xgb_pred = xgb_model.predict(X_test)

print(classification_report(y_test, xgb_pred, target_names=["Normal (0)", "Fraud (1)"]))

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test, xgb_pred, display_labels=["Normal", "Fraud"], cmap="Oranges"
)
plt.title("XGBoost - Confusion Matrix")
plt.savefig("figures/xgb_confusion.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
RocCurveDisplay.from_estimator(xgb_model, X_test, y_test, name="XGBoost")
plt.plot([0, 1], [0, 1], "k--", label="Random Chance")
plt.title("XGBoost - ROC Curve")
plt.legend(loc="lower right")
plt.grid(True, linestyle="--", alpha=0.5)
plt.savefig("figures/xgb_roc.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
xgb_importance = pd.Series(xgb_model.feature_importances_, index=X.columns)

xgb_importance.sort_values().tail(10).plot(kind="barh", color="darkorange")
plt.title("XGBoost - Top 10 Feature Importance")
plt.xlabel("Importance")
plt.savefig("figures/xgb_importance.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
from xgboost import plot_tree as xgb_plot_tree

plt.figure(figsize=(24, 12))
xgb_plot_tree(xgb_model, tree_idx=0, rankdir="LR")
plt.title("XGBoost Tree 1")
plt.savefig("figures/xgb_tree.png", dpi=200, bbox_inches="tight")
plt.show()

## 7. เปรียบเทียบทั้ง 3 โมเดล

In [ ]:
summary = pd.DataFrame({
    "Decision Tree": [accuracy_score(y_test, dt_pred), precision_score(y_test, dt_pred),
                      recall_score(y_test, dt_pred), f1_score(y_test, dt_pred)],
    "Random Forest": [accuracy_score(y_test, rf_pred), precision_score(y_test, rf_pred),
                      recall_score(y_test, rf_pred), f1_score(y_test, rf_pred)],
    "XGBoost": [accuracy_score(y_test, xgb_pred), precision_score(y_test, xgb_pred),
                recall_score(y_test, xgb_pred), f1_score(y_test, xgb_pred)],
}, index=["Accuracy", "Precision", "Recall", "F1-score"]).round(4)

summary

In [ ]:
summary.drop(index="Accuracy").plot(kind="bar", figsize=(10, 6), rot=0,
                                    color=["steelblue", "seagreen", "darkorange"])
plt.title("Model Comparison (Fraud = class 1)")
plt.ylabel("Score")
plt.ylim(0, 1)
plt.legend(loc="upper right")
plt.grid(axis="y", linestyle="--", alpha=0.5)
plt.savefig("figures/compare_metrics.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
plt.figure(figsize=(7, 6))

dt_proba = dt_model.predict_proba(X_test)[:, 1]
rf_proba = rf_model.predict_proba(X_test)[:, 1]
xgb_proba = xgb_model.predict_proba(X_test)[:, 1]

for name, proba, color in [("Decision Tree", dt_proba, "steelblue"),
                           ("Random Forest", rf_proba, "seagreen"),
                           ("XGBoost", xgb_proba, "darkorange")]:
    fpr, tpr, _ = roc_curve(y_test, proba)
    plt.plot(fpr, tpr, color=color, label=f"{name} (AUC = {roc_auc_score(y_test, proba):.3f})")

plt.plot([0, 1], [0, 1], "k--", label="Random Chance")
plt.title("ROC Curve Comparison")
plt.legend(loc="lower right")
plt.grid(True, linestyle="--", alpha=0.5)
plt.savefig("figures/compare_roc.png", dpi=200, bbox_inches="tight")
plt.show()